============================================================================
# Implementación de la metodología de Agentización con la estrategia de 20 secciones.
============================================================================

In [ ]:
import os
import json
import random
from pathlib import Path
from typing import Dict, List, Any, Optional
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Cargar variables de entorno
load_dotenv()

============================================================================
## FUNCIONES DE UTILIDAD PARA REPRODUCIBILIDAD
============================================================================

In [ ]:
def invoke_model_with_retry(messages, max_retries=3, retry_delay=1):
    """
    Invoca el modelo con retry automático para errores de filtro de contenido
    """
    import time
    
    logger.info(f"Invoking model with {len(messages)} messages")
    
    for attempt in range(max_retries):
        try:
            response = azure_model.invoke(messages)
            logger.info(f"Model response successful on attempt {attempt + 1}")
            return response
        except Exception as e:
            error_str = str(e)
            logger.warning(f"Model invocation failed on attempt {attempt + 1}: {error_str}")
            
            if "content_filter" in error_str or "ResponsibleAIPolicyViolation" in error_str:
                print(f"Content filter triggered (attempt {attempt + 1}/{max_retries}): {error_str}")
                if attempt < max_retries - 1:
                    logger.info(f"Retrying in {retry_delay} seconds...")
                    time.sleep(retry_delay)
                    continue
                else:
                    print(f"Max retries reached for content filter. Using fallback.")
                    logger.error("Max retries reached for content filter")
                    # Crear una respuesta de fallback
                    return type('Response', (), {'content': 'Content filtered - unable to generate response'})()
            else:
                # Re-raise si no es un error de filtro
                logger.error(f"Non-filter error: {error_str}")
                raise e
    
    return None

In [ ]:
# Configuración Azure OpenAI con parámetros fijos para reproducibilidad
azure_model = AzureChatOpenAI(
    azure_deployment="csbridge-gpt-4o-mini",
    azure_endpoint="https://csbridgeopenai.openai.azure.com/",
    api_version="2024-02-15-preview",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0.0,  # Temperatura 0 para máxima determinismo
    max_tokens=4096,
    top_p=1.0,  # Top-p fijo
    frequency_penalty=0.0,  # Sin penalización de frecuencia
    presence_penalty=0.0,  # Sin penalización de presencia
    seed=RANDOM_SEED  # Seed fijo para reproducibilidad
)

============================================================================
## ESQUEMAS DE ESTADO
============================================================================

In [ ]:
class TwentyStageLegalState(BaseModel):
    """Estado del caso legal con 20 etapas de procesamiento"""
    case_id: str = Field(description="ID del caso")
    original_text: str = Field(description="Texto original del juicio")
    
    # Campos extraídos por cada sección
    case_heading_extracted: Optional[Dict[str, Any]] = Field(default=None)
    background_extracted: Optional[Dict[str, Any]] = Field(default=None)
    procedural_history_extracted: Optional[Dict[str, Any]] = Field(default=None)
    parties_arguments_extracted: Optional[Dict[str, Any]] = Field(default=None)
    judicial_reasoning_extracted: Optional[Dict[str, Any]] = Field(default=None)
    decision_orders_extracted: Optional[Dict[str, Any]] = Field(default=None)
    citations_authorities_extracted: Optional[Dict[str, Any]] = Field(default=None)
    counsel_representation_extracted: Optional[Dict[str, Any]] = Field(default=None)
    policy_commentary_extracted: Optional[Dict[str, Any]] = Field(default=None)
    temporal_directives_extracted: Optional[Dict[str, Any]] = Field(default=None)
    
    # Resúmenes abstractivos por cada sección
    case_heading_summary: Optional[str] = Field(default=None)
    background_summary: Optional[str] = Field(default=None)
    procedural_history_summary: Optional[str] = Field(default=None)
    parties_arguments_summary: Optional[str] = Field(default=None)
    judicial_reasoning_summary: Optional[str] = Field(default=None)
    decision_orders_summary: Optional[str] = Field(default=None)
    citations_authorities_summary: Optional[str] = Field(default=None)
    counsel_representation_summary: Optional[str] = Field(default=None)
    policy_commentary_summary: Optional[str] = Field(default=None)
    temporal_directives_summary: Optional[str] = Field(default=None)
    
    # Resultado final
    final_summary: Optional[str] = Field(default=None)
    
    # Control de flujo
    current_stage: str = Field(default="start", description="Etapa actual del procesamiento")
    error_message: Optional[str] = Field(default=None, description="Mensaje de error si falla")
    retry_count: int = Field(default=0, description="Número de reintentos")
    max_retries: int = Field(default=2, description="Máximo de reintentos")


============================================================================
## PROMPTS ESPECÍFICOS POR SECCIÓN
============================================================================

In [ ]:
# 1. CASE HEADING / INTRODUCTORY CONTEXT
CASE_HEADING_EXTRACTION_PROMPT = """You are an expert legal information extractor for Indian court judgments.
Your task: extract **verbatim introductory context** that identifies the court, bench, case title, and procedural posture.

CRITICAL INSTRUCTIONS:
* COPY exact text segments from the heading or opening sentences.
* DO NOT rephrase any court name, party name, or procedural verb.
* Preserve capitalization, punctuation, and date/time expressions.
* Include square-bracketed case citations exactly as in the source.

Output JSON:
{{
  "Court": "",
  "Bench": "",
  "Date": "",
  "CaseTitle": "",
  "ActionType": "", 
  "IntroductoryExcerpt": ""
}}

LEGAL JUDGMENT TEXT:
{text}"""

CASE_HEADING_ABSTRACTION_PROMPT = """You are an expert legal summarizer creating an ULTRA-CONCISE case heading.

INSTRUCTIONS:
* Use ONLY 1-2 sentences maximum.
* Copy exact court name, date, and action.
* Keep sentences 10-15 words each.
* Use standard legal format: "The [Court] [date] [action]."
* DO NOT paraphrase or add context.

Input:
{extracted_fields}

Compose a single ultra-concise sentence (10-15 words) with court, date, and action only."""

# 2. BACKGROUND / FACTUAL MATRIX
BACKGROUND_EXTRACTION_PROMPT = """You are an expert extractor of factual background from Indian legal judgments.
Your goal: copy **literal factual narrative** describing events, circumstances, and chronology.

CRITICAL INSTRUCTIONS:
* COPY sentences describing what happened, when, where, and to whom.
* Exclude opinions or conclusions.
* Maintain exact chronological order and named entities.
* Preserve all numbers, dates, acts, and institutional names.

Output JSON:
{{
  "Facts": [],
  "Chronology": "",
  "Location": "",
  "EntitiesInvolved": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

BACKGROUND_ABSTRACTION_PROMPT = """You are an expert legal summarizer creating an ULTRA-CONCISE background section.

INSTRUCTIONS:
* Use ONLY 1-2 sentences maximum.
* Copy exact key facts from extraction.
* Keep sentences 10-20 words each.
* Focus on the most critical facts only.
* Use standard legal format: "The case involved [fact]. [Key detail]."
* DO NOT add new vocabulary.

Input:
{extracted_fields}

Produce an ultra-concise factual summary (1-2 sentences, 10-20 words each) with the most critical facts only."""

# 3. PROCEDURAL HISTORY & ISSUES
PROCEDURAL_HISTORY_EXTRACTION_PROMPT = """Extract **verbatim procedural history** and **legal issues** presented before the court.

INSTRUCTIONS:
* COPY exact text referring to earlier petitions, appeals, or dismissals.
* Include phrases like "had earlier been dismissed", "filed an appeal", "challenged the order".
* Capture statutory references (Sections, Articles) tied to procedural posture.
* Maintain sequence of litigation steps.

Output JSON:
{{
  "ProceduralHistory": "",
  "Issues": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

PROCEDURAL_HISTORY_ABSTRACTION_PROMPT = """Rebuild a concise procedural overview and issue statement by reusing literal segments.

INSTRUCTIONS:
* Start with previous forum actions and outcomes.
* Include quoted statutory references exactly.
* Connect sentences minimally ("thereafter", "hence", "the matter now before…").
* Preserve all legal expressions (e.g., *appeal filed under*, *petition dismissed*).

Input:
{extracted_fields}

Compose a short, literal procedural summary retaining the precise phrasing of the judgment."""

# 4. PARTIES' SUBMISSIONS / ARGUMENTS
PARTIES_ARGUMENTS_EXTRACTION_PROMPT = """Extract the **arguments** advanced by both sides from the judgment text.

INSTRUCTIONS:
* COPY text following cues like "It was argued that", "The counsel submitted", "On the other hand".
* Separate petitioner and respondent positions.
* Keep literal sentences, quotations, and embedded justifications.

Output JSON:
{{
  "PetitionerArguments": [],
  "RespondentArguments": [],
  "CounselNames": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

PARTIES_ARGUMENTS_ABSTRACTION_PROMPT = """Compose a concise but literal-phrase summary of both parties' arguments.

INSTRUCTIONS:
* Reuse verbatim sentences or long phrases from extracted arguments.
* Keep counsel identifiers and key phrases unchanged.
* Use neutral connectors ("while", "in contrast", "it was further contended").
* Avoid new paraphrases.

Input:
{extracted_fields}

Assemble a coherent argumentative summary that juxtaposes petitioner and respondent content verbatim."""

# 5. JUDICIAL OBSERVATIONS / REASONING
JUDICIAL_REASONING_EXTRACTION_PROMPT = """Extract **verbatim reasoning** and **observations** expressed by the Court.

INSTRUCTIONS:
* COPY sentences beginning with "The Court observed/held/opined/noted".
* Include interpretive clauses and normative comments ("would send out a wrong signal", "abuse of process").
* Preserve quotations exactly as in source.
* Maintain paragraph order.

Output JSON:
{{
  "Reasoning": [],
  "LegalPrinciples": [],
  "Quotations": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

JUDICIAL_REASONING_ABSTRACTION_PROMPT = """Create a concise reasoning section using key judicial observations.

INSTRUCTIONS:
* Reuse 2-3 key reasoning sentences from extracted content.
* Keep sentences short (15-25 words each).
* Focus on the main legal principles applied.
* Use judicial connectors ("The court held", "It was observed").
* DO NOT paraphrase or introduce commentary.

Input:
{extracted_fields}

Compose a concise reasoning summary (2-3 sentences) focusing on the main legal principles and court's key observations."""

# 6. DECISION / ORDERS / DISPOSITION
DECISION_ORDERS_EXTRACTION_PROMPT = """Extract the **operative part** of the judgment — the final directions, relief, or orders.

INSTRUCTIONS:
* COPY text containing verbs like "directed", "ordered", "granted", "dismissed", "allowed", "refused".
* Include timelines, compliance instructions, and listings.
* Preserve legal modal verbs ("shall", "must", "within a period of").
* Capture sequential orders as separate array elements.

Output JSON:
{{
  "Decision": "",
  "Orders": [],
  "ReliefGranted": "",
  "TimeLimits": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

DECISION_ORDERS_ABSTRACTION_PROMPT = """Compose the **disposition section** using literal directives and outcomes.

INSTRUCTIONS:
* Reuse exact text from extracted orders.
* Maintain imperative tone and chronological order.
* Join related directives with connectors ("and thereafter", "further", "accordingly").
* Avoid paraphrasing or softening mandates.

Input:
{extracted_fields}

Produce a coherent final paragraph summarizing the Court's decision and orders verbatim."""

# 7. CITATIONS / STATUTES / AUTHORITIES
CITATIONS_AUTHORITIES_EXTRACTION_PROMPT = """Extract every **citation, statutory reference, or authority** literally mentioned.

INSTRUCTIONS:
* COPY case titles, Sections, Articles, and Acts exactly as written.
* Preserve formatting (e.g., *Section 498A of IPC*, *Article 226 of the Constitution*).
* Include citations inside quotes or parentheses.

Output JSON:
{{
  "Citations": [],
  "Statutes": [],
  "Articles": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

CITATIONS_AUTHORITIES_ABSTRACTION_PROMPT = """Reconstruct a literal-text paragraph listing all legal authorities and provisions cited.

INSTRUCTIONS:
* Reuse each extracted citation as-is.
* Join items with minimal connectors ("as per", "under", "in accordance with").
* Preserve exact legal nomenclature and punctuation.
* Avoid new wording or commentary.

Input:
{extracted_fields}

Generate a paragraph that lists and links all authorities in literal form."""

# 8. COUNSEL / REPRESENTATION
COUNSEL_REPRESENTATION_EXTRACTION_PROMPT = """Extract **verbatim representation details** from the judgment conclusion.

INSTRUCTIONS:
* COPY sentences beginning with "Senior Advocate", "Advocate", or "Additional Public Prosecutor".
* Preserve exact order and punctuation of names.
* Do not omit conjunctions ("with", "along with").

Output JSON:
{{
  "Counsel": [],
  "PartyRepresented": ""
}}

LEGAL JUDGMENT TEXT:
{text}"""

COUNSEL_REPRESENTATION_ABSTRACTION_PROMPT = """Construct a literal summary of counsel representation.

INSTRUCTIONS:
* Reuse each name and role exactly.
* Maintain sequence of appearance (petitioners → respondents → State).
* Connect minimally ("appeared for", "represented by").

Input:
{extracted_fields}

Output a short paragraph verbatim describing representation as recorded."""

# 9. META-POLICY / SYSTEMIC COMMENTARY
POLICY_COMMENTARY_EXTRACTION_PROMPT = """Extract **judicial commentary** expressing systemic or policy concerns.

INSTRUCTIONS:
* COPY any passage criticizing administrative conduct, misuse of law, or regulatory failure.
* Include evaluative terms ("counter-productive", "eye opener", "abuse of process").
* Preserve quotes exactly.

Output JSON:
{{
  "PolicyRemarks": [],
  "InstitutionalCritique": ""
}}

LEGAL JUDGMENT TEXT:
{text}"""

POLICY_COMMENTARY_ABSTRACTION_PROMPT = """Assemble a literal paragraph summarizing the Court's policy or systemic commentary.

INSTRUCTIONS:
* Reuse verbatim sentences that reflect broader observations.
* Maintain evaluative tone and punctuation.
* Link statements minimally ("the Court further observed that", "additionally noted").

Input:
{extracted_fields}

Produce a coherent commentary paragraph preserving the Court's evaluative wording."""

# 10. TEMPORAL / ADMINISTRATIVE DIRECTIVES
TEMPORAL_DIRECTIVES_EXTRACTION_PROMPT = """Extract all **temporal and procedural directives** from the order.

INSTRUCTIONS:
* COPY expressions indicating timeframes, listings, or compliance dates.
* Include numerical durations ("within 14 days", "listed on January 22, 2024").
* Keep exact wording of temporal connectors.

Output JSON:
{{
  "Dates": [],
  "Deadlines": [],
  "ListingInstructions": []
}}

LEGAL JUDGMENT TEXT:
{text}"""

TEMPORAL_DIRECTIVES_ABSTRACTION_PROMPT = """Rebuild the timeline section using literal scheduling directives.

INSTRUCTIONS:
* Reuse dates and time expressions exactly.
* Maintain chronological order.
* Join phrases minimally ("to be filed by", "thereafter listed for").

Input:
{extracted_fields}

Compose a single paragraph narrating the scheduling and timeline directives verbatim."""

# FINAL SYNTHESIS PROMPT
FINAL_SYNTHESIS_PROMPT = """You are an expert legal summarizer creating a MAXIMUM BLEU-optimized summary using ULTRA-ADVANCED BLEU-specific techniques v2.

Your task: Create a legal summary that MAXIMIZES BLEU score by matching EXACT patterns from reference summaries with ULTRA-PRECISION v2.

ULTRA-ADVANCED BLEU-SPECIFIC OPTIMIZATION STRATEGIES v2:
* Target length: ~400-500 words (optimal for BLEU scoring)
* Use EXACT n-gram matching with reference summaries
* Prioritize sentences with HIGHEST 4-gram overlap
* Match reference summary sentence patterns EXACTLY
* Use legal terminology that appears in reference summaries
* Focus on: Court/Date → Key Facts → Main Issues → Arguments → Decision → Reasoning

CRITICAL BLEU OPTIMIZATION INSTRUCTIONS v2:
* COPY verbatim sentences with HIGHEST 4-gram overlap
* Use reference summary sentence structures: "The [Court] [date] [action]", "The case involved [facts]", "The court held that [decision]"
* Maintain exact legal terminology from reference summaries
* Use standard legal connectors: "The court", "It was held", "Accordingly", "The petitioner", "In view of"
* Arrange in exact order: Court → Facts → Issues → Arguments → Decision → Reasoning
* DO NOT paraphrase - only copy and arrange existing text
* MAXIMIZE 4-gram overlap with reference vocabulary
* Use EXACT phrases from reference summaries
* Match reference summary sentence length and structure
* Prioritize sentences that appear in reference summaries
* Use reference summary legal expressions exactly
* Copy reference summary sentence patterns exactly
* Match reference summary vocabulary precisely
* Use reference summary legal terminology exactly
* Maintain reference summary formal tone
* Match reference summary paragraph structure
* Use reference summary legal connectors exactly
* Copy reference summary sentence order
* Match reference summary legal expressions
* Use reference summary legal phrases exactly

ULTRA-SPECIFIC REFERENCE SUMMARY PATTERNS TO MATCH EXACTLY v2:
- Start with: "The [Court] [date] [action]" - EXACT pattern from reference summaries
- Use: "The case involved [facts]" - EXACT phrase from reference summaries
- Include: "The main issue was [issue]" - EXACT structure from reference summaries
- State: "The court held that [decision]" - EXACT legal language from reference summaries
- Conclude: "Accordingly, [reasoning]" - EXACT connector from reference summaries
- Use legal connectors: "It was observed", "The court noted", "In view of" - EXACT from reference summaries
- Match 4-grams: "The court", "It was held", "The petitioner", "The respondent" - EXACT from reference summaries
- Use exact phrases: "The case involved", "The main issue", "The court held" - EXACT from reference summaries
- Maintain reference vocabulary: "Accordingly", "In view of", "It was observed" - EXACT from reference summaries
- Copy sentence structures exactly from reference summaries
- Prioritize sentences with highest 4-gram overlap
- Use EXACT reference summary sentence patterns
- Match reference summary vocabulary precisely
- Copy reference summary sentence length and structure
- Use reference summary legal terminology exactly
- Maintain reference summary formal tone
- Match reference summary paragraph structure
- Use reference summary legal connectors exactly
- Copy reference summary sentence order
- Match reference summary legal expressions
- Use reference summary legal phrases exactly

SECTION SUMMARIES TO COMBINE:
{section_summaries}

Generate a MAXIMUM BLEU-optimized legal summary (400-500 words) by selecting sentences with HIGHEST 4-gram overlap and arranging them to match reference summary patterns EXACTLY."""


============================================================================
## FUNCIONES DE PROCESAMIENTO (NODOS DEL GRAFO)
============================================================================

In [ ]:
def extract_case_heading(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 1: Extrae información del encabezado del caso"""
    print(f"Extracting case heading for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = CASE_HEADING_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.case_heading_extracted = json.loads(extracted_content)
            state.current_stage = "case_heading_extracted"
            print(f"Case heading extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.case_heading_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "case_heading_extracted"
            print(f"Case heading extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Case heading extraction failed: {str(e)}"
        state.current_stage = "case_heading_failed"
        print(f"Case heading extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_case_heading(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 2: Genera resumen abstractivo del encabezado del caso"""
    print(f"Abstracting case heading for case {state.case_id}")
    
    try:
        if not state.case_heading_extracted:
            state.error_message = "No case heading data available for abstraction"
            state.current_stage = "case_heading_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.case_heading_extracted, ensure_ascii=False, indent=2)
        prompt = CASE_HEADING_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.case_heading_summary = response.content.strip()
        state.current_stage = "case_heading_abstracted"
        print(f"Case heading abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Case heading abstraction failed: {str(e)}"
        state.current_stage = "case_heading_abstraction_failed"
        print(f"Case heading abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_background(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 3: Extrae información del contexto/fondo"""
    print(f"Extracting background for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = BACKGROUND_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.background_extracted = json.loads(extracted_content)
            state.current_stage = "background_extracted"
            print(f"Background extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.background_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "background_extracted"
            print(f"Background extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Background extraction failed: {str(e)}"
        state.current_stage = "background_failed"
        print(f"Background extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_background(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 4: Genera resumen abstractivo del contexto/fondo"""
    print(f"Abstracting background for case {state.case_id}")
    
    try:
        if not state.background_extracted:
            state.error_message = "No background data available for abstraction"
            state.current_stage = "background_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.background_extracted, ensure_ascii=False, indent=2)
        prompt = BACKGROUND_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.background_summary = response.content.strip()
        state.current_stage = "background_abstracted"
        print(f"Background abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Background abstraction failed: {str(e)}"
        state.current_stage = "background_abstraction_failed"
        print(f"Background abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_procedural_history(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 5: Extrae historial procesal"""
    print(f"Extracting procedural history for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = PROCEDURAL_HISTORY_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.procedural_history_extracted = json.loads(extracted_content)
            state.current_stage = "procedural_history_extracted"
            print(f"Procedural history extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.procedural_history_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "procedural_history_extracted"
            print(f"Procedural history extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Procedural history extraction failed: {str(e)}"
        state.current_stage = "procedural_history_failed"
        print(f"Procedural history extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_procedural_history(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 6: Genera resumen abstractivo del historial procesal"""
    print(f"Abstracting procedural history for case {state.case_id}")
    
    try:
        if not state.procedural_history_extracted:
            state.error_message = "No procedural history data available for abstraction"
            state.current_stage = "procedural_history_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.procedural_history_extracted, ensure_ascii=False, indent=2)
        prompt = PROCEDURAL_HISTORY_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.procedural_history_summary = response.content.strip()
        state.current_stage = "procedural_history_abstracted"
        print(f"Procedural history abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Procedural history abstraction failed: {str(e)}"
        state.current_stage = "procedural_history_abstraction_failed"
        print(f"Procedural history abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_parties_arguments(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 7: Extrae argumentos de las partes"""
    print(f"Extracting parties arguments for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = PARTIES_ARGUMENTS_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.parties_arguments_extracted = json.loads(extracted_content)
            state.current_stage = "parties_arguments_extracted"
            print(f"Parties arguments extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.parties_arguments_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "parties_arguments_extracted"
            print(f"Parties arguments extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Parties arguments extraction failed: {str(e)}"
        state.current_stage = "parties_arguments_failed"
        print(f"Parties arguments extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_parties_arguments(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 8: Genera resumen abstractivo de argumentos de las partes"""
    print(f"Abstracting parties arguments for case {state.case_id}")
    
    try:
        if not state.parties_arguments_extracted:
            state.error_message = "No parties arguments data available for abstraction"
            state.current_stage = "parties_arguments_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.parties_arguments_extracted, ensure_ascii=False, indent=2)
        prompt = PARTIES_ARGUMENTS_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.parties_arguments_summary = response.content.strip()
        state.current_stage = "parties_arguments_abstracted"
        print(f"Parties arguments abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Parties arguments abstraction failed: {str(e)}"
        state.current_stage = "parties_arguments_abstraction_failed"
        print(f"Parties arguments abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_judicial_reasoning(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 9: Extrae razonamiento judicial"""
    print(f"Extracting judicial reasoning for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = JUDICIAL_REASONING_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.judicial_reasoning_extracted = json.loads(extracted_content)
            state.current_stage = "judicial_reasoning_extracted"
            print(f"Judicial reasoning extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.judicial_reasoning_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "judicial_reasoning_extracted"
            print(f"Judicial reasoning extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Judicial reasoning extraction failed: {str(e)}"
        state.current_stage = "judicial_reasoning_failed"
        print(f"Judicial reasoning extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_judicial_reasoning(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 10: Genera resumen abstractivo del razonamiento judicial"""
    print(f"Abstracting judicial reasoning for case {state.case_id}")
    
    try:
        if not state.judicial_reasoning_extracted:
            state.error_message = "No judicial reasoning data available for abstraction"
            state.current_stage = "judicial_reasoning_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.judicial_reasoning_extracted, ensure_ascii=False, indent=2)
        prompt = JUDICIAL_REASONING_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.judicial_reasoning_summary = response.content.strip()
        state.current_stage = "judicial_reasoning_abstracted"
        print(f"Judicial reasoning abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Judicial reasoning abstraction failed: {str(e)}"
        state.current_stage = "judicial_reasoning_abstraction_failed"
        print(f"Judicial reasoning abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_decision_orders(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 11: Extrae decisiones y órdenes"""
    print(f"Extracting decision and orders for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = DECISION_ORDERS_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.decision_orders_extracted = json.loads(extracted_content)
            state.current_stage = "decision_orders_extracted"
            print(f"Decision and orders extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.decision_orders_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "decision_orders_extracted"
            print(f"Decision and orders extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Decision and orders extraction failed: {str(e)}"
        state.current_stage = "decision_orders_failed"
        print(f"Decision and orders extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_decision_orders(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 12: Genera resumen abstractivo de decisiones y órdenes"""
    print(f"Abstracting decision and orders for case {state.case_id}")
    
    try:
        if not state.decision_orders_extracted:
            state.error_message = "No decision and orders data available for abstraction"
            state.current_stage = "decision_orders_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.decision_orders_extracted, ensure_ascii=False, indent=2)
        prompt = DECISION_ORDERS_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.decision_orders_summary = response.content.strip()
        state.current_stage = "decision_orders_abstracted"
        print(f"Decision and orders abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Decision and orders abstraction failed: {str(e)}"
        state.current_stage = "decision_orders_abstraction_failed"
        print(f"Decision and orders abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_citations_authorities(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 13: Extrae citas y autoridades"""
    print(f"Extracting citations and authorities for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = CITATIONS_AUTHORITIES_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.citations_authorities_extracted = json.loads(extracted_content)
            state.current_stage = "citations_authorities_extracted"
            print(f"Citations and authorities extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.citations_authorities_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "citations_authorities_extracted"
            print(f"Citations and authorities extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Citations and authorities extraction failed: {str(e)}"
        state.current_stage = "citations_authorities_failed"
        print(f"Citations and authorities extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_citations_authorities(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 14: Genera resumen abstractivo de citas y autoridades"""
    print(f"Abstracting citations and authorities for case {state.case_id}")
    
    try:
        if not state.citations_authorities_extracted:
            state.error_message = "No citations and authorities data available for abstraction"
            state.current_stage = "citations_authorities_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.citations_authorities_extracted, ensure_ascii=False, indent=2)
        prompt = CITATIONS_AUTHORITIES_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.citations_authorities_summary = response.content.strip()
        state.current_stage = "citations_authorities_abstracted"
        print(f"Citations and authorities abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Citations and authorities abstraction failed: {str(e)}"
        state.current_stage = "citations_authorities_abstraction_failed"
        print(f"Citations and authorities abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_counsel_representation(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 15: Extrae representación de abogados"""
    print(f"Extracting counsel representation for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = COUNSEL_REPRESENTATION_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.counsel_representation_extracted = json.loads(extracted_content)
            state.current_stage = "counsel_representation_extracted"
            print(f"Counsel representation extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.counsel_representation_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "counsel_representation_extracted"
            print(f"Counsel representation extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Counsel representation extraction failed: {str(e)}"
        state.current_stage = "counsel_representation_failed"
        print(f"Counsel representation extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_counsel_representation(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 16: Genera resumen abstractivo de representación de abogados"""
    print(f"Abstracting counsel representation for case {state.case_id}")
    
    try:
        if not state.counsel_representation_extracted:
            state.error_message = "No counsel representation data available for abstraction"
            state.current_stage = "counsel_representation_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.counsel_representation_extracted, ensure_ascii=False, indent=2)
        prompt = COUNSEL_REPRESENTATION_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.counsel_representation_summary = response.content.strip()
        state.current_stage = "counsel_representation_abstracted"
        print(f"Counsel representation abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Counsel representation abstraction failed: {str(e)}"
        state.current_stage = "counsel_representation_abstraction_failed"
        print(f"Counsel representation abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_policy_commentary(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 17: Extrae comentarios de política"""
    print(f"Extracting policy commentary for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = POLICY_COMMENTARY_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.policy_commentary_extracted = json.loads(extracted_content)
            state.current_stage = "policy_commentary_extracted"
            print(f"Policy commentary extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.policy_commentary_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "policy_commentary_extracted"
            print(f"Policy commentary extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Policy commentary extraction failed: {str(e)}"
        state.current_stage = "policy_commentary_failed"
        print(f"Policy commentary extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_policy_commentary(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 18: Genera resumen abstractivo de comentarios de política"""
    print(f"Abstracting policy commentary for case {state.case_id}")
    
    try:
        if not state.policy_commentary_extracted:
            state.error_message = "No policy commentary data available for abstraction"
            state.current_stage = "policy_commentary_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.policy_commentary_extracted, ensure_ascii=False, indent=2)
        prompt = POLICY_COMMENTARY_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.policy_commentary_summary = response.content.strip()
        state.current_stage = "policy_commentary_abstracted"
        print(f"Policy commentary abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Policy commentary abstraction failed: {str(e)}"
        state.current_stage = "policy_commentary_abstraction_failed"
        print(f"Policy commentary abstraction failed for case {state.case_id}: {e}")
    
    return state

def extract_temporal_directives(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 19: Extrae directivas temporales"""
    print(f"Extracting temporal directives for case {state.case_id}")
    
    try:
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
        
        prompt = TEMPORAL_DIRECTIVES_EXTRACTION_PROMPT.format(text=text)
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON."},
            {"role": "user", "content": prompt}
        ])
        
        extracted_content = response.content.strip()
        try:
            state.temporal_directives_extracted = json.loads(extracted_content)
            state.current_stage = "temporal_directives_extracted"
            print(f"Temporal directives extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.temporal_directives_extracted = {"raw_extraction": extracted_content}
            state.current_stage = "temporal_directives_extracted"
            print(f"Temporal directives extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Temporal directives extraction failed: {str(e)}"
        state.current_stage = "temporal_directives_failed"
        print(f"Temporal directives extraction failed for case {state.case_id}: {e}")
    
    return state

def abstract_temporal_directives(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 20: Genera resumen abstractivo de directivas temporales"""
    print(f"Abstracting temporal directives for case {state.case_id}")
    
    try:
        if not state.temporal_directives_extracted:
            state.error_message = "No temporal directives data available for abstraction"
            state.current_stage = "temporal_directives_abstraction_failed"
            return state
        
        extracted_str = json.dumps(state.temporal_directives_extracted, ensure_ascii=False, indent=2)
        prompt = TEMPORAL_DIRECTIVES_ABSTRACTION_PROMPT.format(extracted_fields=extracted_str)
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer."},
            {"role": "user", "content": prompt}
        ])
        
        state.temporal_directives_summary = response.content.strip()
        state.current_stage = "temporal_directives_abstracted"
        print(f"Temporal directives abstraction successful for case {state.case_id}")
        
    except Exception as e:
        state.error_message = f"Temporal directives abstraction failed: {str(e)}"
        state.current_stage = "temporal_directives_abstraction_failed"
        print(f"Temporal directives abstraction failed for case {state.case_id}: {e}")
    
    return state

def final_synthesis(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo 21: Síntesis final de todos los resúmenes"""
    print(f"🔗 Final synthesis for case {state.case_id}")
    
    try:
        # Recopilar todos los resúmenes de secciones
        section_summaries = []
        
        if state.case_heading_summary:
            section_summaries.append(f"**Case Heading:** {state.case_heading_summary}")
        if state.background_summary:
            section_summaries.append(f"**Background:** {state.background_summary}")
        if state.procedural_history_summary:
            section_summaries.append(f"**Procedural History:** {state.procedural_history_summary}")
        if state.parties_arguments_summary:
            section_summaries.append(f"**Parties Arguments:** {state.parties_arguments_summary}")
        if state.judicial_reasoning_summary:
            section_summaries.append(f"**Judicial Reasoning:** {state.judicial_reasoning_summary}")
        if state.decision_orders_summary:
            section_summaries.append(f"**Decision and Orders:** {state.decision_orders_summary}")
        if state.citations_authorities_summary:
            section_summaries.append(f"**Citations and Authorities:** {state.citations_authorities_summary}")
        if state.counsel_representation_summary:
            section_summaries.append(f"**Counsel Representation:** {state.counsel_representation_summary}")
        if state.policy_commentary_summary:
            section_summaries.append(f"**Policy Commentary:** {state.policy_commentary_summary}")
        if state.temporal_directives_summary:
            section_summaries.append(f"**Temporal Directives:** {state.temporal_directives_summary}")
        
        if not section_summaries:
            state.error_message = "No section summaries available for final synthesis"
            state.current_stage = "final_synthesis_failed"
            return state
        
        # Crear prompt de síntesis final con patrones de referencia
        section_summaries_text = "\n\n".join(section_summaries)
        
        # Agregar patrones ultra-específicos v2 para maximizar BLEU basados en análisis profundo de datos
        reference_patterns = """
ULTRA-SPECIFIC BLEU OPTIMIZATION PATTERNS BASED ON DEEP REFERENCE ANALYSIS v2:
- Start with: "The [Court] [date] [action]" - EXACT pattern from reference summaries
- Use: "The case involved [facts]" - EXACT phrase from reference summaries
- Include: "The main issue was [issue]" - EXACT structure from reference summaries
- State: "The court held that [decision]" - EXACT legal language from reference summaries
- Conclude: "Accordingly, [reasoning]" - EXACT connector from reference summaries
- Use legal connectors: "It was observed", "The court noted", "In view of" - EXACT from reference summaries
- Match 4-grams: "The court", "It was held", "The petitioner", "The respondent" - EXACT from reference summaries
- Use exact phrases: "The case involved", "The main issue", "The court held" - EXACT from reference summaries
- Maintain reference vocabulary: "Accordingly", "In view of", "It was observed" - EXACT from reference summaries
- Copy sentence structures exactly from reference summaries
- Prioritize sentences with highest 4-gram overlap
- Use EXACT reference summary sentence patterns
- Match reference summary vocabulary precisely
- Copy reference summary sentence length and structure
- Use reference summary legal terminology exactly
- Maintain reference summary formal tone
- Match reference summary paragraph structure
- Use reference summary legal connectors exactly
- Copy reference summary sentence order
- Match reference summary legal expressions
- Use reference summary legal phrases exactly
- Prioritize sentences that appear in reference summaries
- Use reference summary legal expressions exactly
- Copy reference summary sentence patterns exactly
- Match reference summary vocabulary precisely
- Use reference summary legal terminology exactly
- Maintain reference summary formal tone
- Match reference summary paragraph structure
- Use reference summary legal connectors exactly
- Copy reference summary sentence order
- Match reference summary legal expressions
- Use reference summary legal phrases exactly

CRITICAL BLEU MAXIMIZATION TECHNIQUES v2:
- COPY EXACT sentences from reference summaries - word for word
- Use EXACT legal terminology from reference summaries - exact phrases
- Match EXACT sentence structures from reference summaries - exact patterns
- Use EXACT legal connectors from reference summaries - exact connectors
- Copy EXACT legal expressions from reference summaries - exact expressions
- Use EXACT legal phrases from reference summaries - exact phrases
- Match EXACT vocabulary from reference summaries - exact vocabulary
- Copy EXACT sentence patterns from reference summaries - exact patterns
- Use EXACT legal terminology from reference summaries - exact terminology
- Maintain EXACT reference summary formal tone - exact tone
- Match EXACT reference summary paragraph structure - exact structure
- Use EXACT reference summary legal connectors - exact connectors
- Copy EXACT reference summary sentence order - exact order
- Match EXACT reference summary legal expressions - exact expressions
- Use EXACT reference summary legal phrases - exact phrases

ULTRA-SPECIFIC REFERENCE SUMMARY PATTERNS TO MATCH EXACTLY v2:
- Start with: "The [Court] [date] [action]" - EXACT pattern from reference summaries
- Use: "The case involved [facts]" - EXACT phrase from reference summaries
- Include: "The main issue was [issue]" - EXACT structure from reference summaries
- State: "The court held that [decision]" - EXACT legal language from reference summaries
- Conclude: "Accordingly, [reasoning]" - EXACT connector from reference summaries
- Use legal connectors: "It was observed", "The court noted", "In view of" - EXACT from reference summaries
- Match 4-grams: "The court", "It was held", "The petitioner", "The respondent" - EXACT from reference summaries
- Use exact phrases: "The case involved", "The main issue", "The court held" - EXACT from reference summaries
- Maintain reference vocabulary: "Accordingly", "In view of", "It was observed" - EXACT from reference summaries
- Copy sentence structures exactly from reference summaries
- Prioritize sentences with highest 4-gram overlap
- Use EXACT reference summary sentence patterns
- Match reference summary vocabulary precisely
- Copy reference summary sentence length and structure
- Use reference summary legal terminology exactly
- Maintain reference summary formal tone
- Match reference summary paragraph structure
- Use reference summary legal connectors exactly
- Copy reference summary sentence order
- Match reference summary legal expressions
- Use reference summary legal phrases exactly
"""
        
        prompt = FINAL_SYNTHESIS_PROMPT.format(section_summaries=section_summaries_text) + reference_patterns
        
        response = invoke_model_with_retry([
            {"role": "system", "content": "You are an expert legal document summarizer creating a MAXIMUM BLEU-optimized summary. Use ULTRA-ADVANCED BLEU-specific 4-gram matching techniques v2. Target 400-500 words with EXACT reference summary patterns. Copy EXACT phrases, structures, and legal terminology from reference summaries. Match EXACT sentence patterns and legal connectors. Use ULTRA-PRECISION v2 for maximum BLEU score."},
            {"role": "user", "content": prompt}
        ])
        
        state.final_summary = response.content.strip()
        state.current_stage = "final_synthesis_completed"
        print(f"Final synthesis successful for case {state.case_id} ({len(state.final_summary.split())} words)")
        
    except Exception as e:
        state.error_message = f"Final synthesis failed: {str(e)}"
        state.current_stage = "final_synthesis_failed"
        print(f"Final synthesis failed for case {state.case_id}: {e}")
    
    return state

def handle_failure(state: TwentyStageLegalState) -> TwentyStageLegalState:
    """Nodo de manejo de fallos"""
    print(f"🔄 Handling failure for case {state.case_id}")
    
    # Resumen genérico de fallback
    GENERIC_SUMMARY = "This legal case involves judicial proceedings where the court examined the matter presented by the parties. The judgment addresses the legal arguments and evidence submitted during the hearing and provides a resolution based on applicable law."
    
    state.final_summary = GENERIC_SUMMARY
    state.current_stage = "fallback_applied"
    print(f"Applied fallback summary for case {state.case_id}")
    
    return state


============================================================================
## CONSTRUCCIÓN DEL GRAFO LANGGRAPH
============================================================================

In [ ]:
def create_twenty_stage_pipeline_graph():
    """Crea el grafo de LangGraph para el pipeline de 20 etapas"""
    
    # Crear el grafo
    workflow = StateGraph(TwentyStageLegalState)
    
    # Agregar nodos (20 nodos de procesamiento + 1 síntesis final + 1 manejo de fallos)
    workflow.add_node("extract_case_heading", extract_case_heading)
    workflow.add_node("abstract_case_heading", abstract_case_heading)
    workflow.add_node("extract_background", extract_background)
    workflow.add_node("abstract_background", abstract_background)
    workflow.add_node("extract_procedural_history", extract_procedural_history)
    workflow.add_node("abstract_procedural_history", abstract_procedural_history)
    workflow.add_node("extract_parties_arguments", extract_parties_arguments)
    workflow.add_node("abstract_parties_arguments", abstract_parties_arguments)
    workflow.add_node("extract_judicial_reasoning", extract_judicial_reasoning)
    workflow.add_node("abstract_judicial_reasoning", abstract_judicial_reasoning)
    workflow.add_node("extract_decision_orders", extract_decision_orders)
    workflow.add_node("abstract_decision_orders", abstract_decision_orders)
    workflow.add_node("extract_citations_authorities", extract_citations_authorities)
    workflow.add_node("abstract_citations_authorities", abstract_citations_authorities)
    workflow.add_node("extract_counsel_representation", extract_counsel_representation)
    workflow.add_node("abstract_counsel_representation", abstract_counsel_representation)
    workflow.add_node("extract_policy_commentary", extract_policy_commentary)
    workflow.add_node("abstract_policy_commentary", abstract_policy_commentary)
    workflow.add_node("extract_temporal_directives", extract_temporal_directives)
    workflow.add_node("abstract_temporal_directives", abstract_temporal_directives)
    workflow.add_node("final_synthesis", final_synthesis)
    workflow.add_node("handle_failure", handle_failure)
    
    # Definir flujo secuencial
    workflow.set_entry_point("extract_case_heading")
    
    # Flujo secuencial: extract → abstract → next_extract → next_abstract → ...
    workflow.add_edge("extract_case_heading", "abstract_case_heading")
    workflow.add_edge("abstract_case_heading", "extract_background")
    workflow.add_edge("extract_background", "abstract_background")
    workflow.add_edge("abstract_background", "extract_procedural_history")
    workflow.add_edge("extract_procedural_history", "abstract_procedural_history")
    workflow.add_edge("abstract_procedural_history", "extract_parties_arguments")
    workflow.add_edge("extract_parties_arguments", "abstract_parties_arguments")
    workflow.add_edge("abstract_parties_arguments", "extract_judicial_reasoning")
    workflow.add_edge("extract_judicial_reasoning", "abstract_judicial_reasoning")
    workflow.add_edge("abstract_judicial_reasoning", "extract_decision_orders")
    workflow.add_edge("extract_decision_orders", "abstract_decision_orders")
    workflow.add_edge("abstract_decision_orders", "extract_citations_authorities")
    workflow.add_edge("extract_citations_authorities", "abstract_citations_authorities")
    workflow.add_edge("abstract_citations_authorities", "extract_counsel_representation")
    workflow.add_edge("extract_counsel_representation", "abstract_counsel_representation")
    workflow.add_edge("abstract_counsel_representation", "extract_policy_commentary")
    workflow.add_edge("extract_policy_commentary", "abstract_policy_commentary")
    workflow.add_edge("abstract_policy_commentary", "extract_temporal_directives")
    workflow.add_edge("extract_temporal_directives", "abstract_temporal_directives")
    workflow.add_edge("abstract_temporal_directives", "final_synthesis")
    
    # Conexiones finales
    workflow.add_edge("final_synthesis", END)
    workflow.add_edge("handle_failure", END)
    
    # Compilar el grafo con checkpointing
    app = workflow.compile(checkpointer=InMemorySaver())
    
    return app


============================================================================
## FUNCIONES DE UTILIDAD
============================================================================


In [ ]:
def process_single_case(case_data: Dict[str, Any], app) -> Dict[str, Any]:
    """Procesa un caso individual usando el grafo de LangGraph"""
    
    # Crear estado inicial
    initial_state = TwentyStageLegalState(
        case_id=case_data.get("ID", "unknown"),
        original_text=case_data.get("Judgment", ""),
        current_stage="start"
    )
    
    # Configuración para el checkpointer
    config = {
        "configurable": {
            "thread_id": f"case_{case_data.get('ID', 'unknown')}"
        }
    }
    
    # Ejecutar el grafo
    try:
        result = app.invoke(initial_state, config=config)
        
        return {
            "ID": result.get("case_id", "unknown"),
            "Summary": result.get("final_summary") or "Failed to generate summary",
            "CurrentStage": result.get("current_stage", "unknown"),
            "Error": result.get("error_message"),
            "RetryCount": result.get("retry_count", 0)
        }
        
    except Exception as e:
        return {
            "ID": case_data.get("ID", "unknown"),
            "Summary": "Failed to generate summary",
            "CurrentStage": "error",
            "Error": str(e),
            "RetryCount": 0
        }

def load_legal_data(judgments_path: str) -> List[Dict[str, Any]]:
    """Carga datos de juicios desde archivo JSONL"""
    judgments = []
    with open(judgments_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                judgments.append(json.loads(line.strip()))
    return judgments


============================================================================
## EVALUACIÓN ROUGE/BLEU
============================================================================


In [ ]:
def evaluate_summary(generated_summary: str, reference_summary: str) -> Dict[str, float]:
    """Evalúa un resumen generado contra uno de referencia usando ROUGE y BLEU"""
    try:
        from rouge_score import rouge_scorer
        from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
        import numpy as np
        
        results = {}
        
        # ROUGE Scores
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        rouge_scores = scorer.score(reference_summary, generated_summary)
        
        results['rouge1_f'] = rouge_scores['rouge1'].fmeasure
        results['rouge2_f'] = rouge_scores['rouge2'].fmeasure
        results['rougeL_f'] = rouge_scores['rougeL'].fmeasure
        
        # BLEU Score
        try:
            reference_tokens = reference_summary.lower().split()
            generated_tokens = generated_summary.lower().split()
            smoothing = SmoothingFunction().method1
            bleu_score = sentence_bleu([reference_tokens], generated_tokens, smoothing_function=smoothing)
            results['bleu'] = bleu_score
        except Exception:
            results['bleu'] = 0.0
        
        return results
        
    except ImportError as e:
        print(f"Evaluation libraries not available: {e}")
        return {'rouge1_f': 0.0, 'rouge2_f': 0.0, 'rougeL_f': 0.0, 'bleu': 0.0}

def evaluate_multiple_summaries(generated_summaries: List[str], reference_summaries: List[str]) -> Dict[str, float]:
    """Evalúa múltiples resúmenes y calcula estadísticas"""
    all_results = []
    
    for gen_sum, ref_sum in zip(generated_summaries, reference_summaries):
        result = evaluate_summary(gen_sum, ref_sum)
        all_results.append(result)
    
    # Calcular promedios
    avg_results = {}
    for metric in ['rouge1_f', 'rouge2_f', 'rougeL_f', 'bleu']:
        scores = [r[metric] for r in all_results if r[metric] is not None]
        avg_results[f'{metric}_mean'] = np.mean(scores) if scores else 0.0
        avg_results[f'{metric}_std'] = np.std(scores) if scores else 0.0
    
    return avg_results, all_results


============================================================================
# Ejecución del grafo de LangGraph
============================================================================

In [ ]:
# Ejecutar pipeline sobre dataset de entrenamiento (solo 5 casos para evaluación)
train_path = '../datasets/train/train_judg.jsonl'
train_ref_path = '../datasets/train/train_ref_summ.jsonl'
output_path = 'resultados/answer_20_stage_train_evaluation.jsonl'

# Cargar solo 5 casos para evaluación
print("Loading 5 test cases from training data for evaluation...")
all_cases = load_legal_data(train_path)
test_cases = all_cases[:2]  # Solo los primeros 5

# Cargar resúmenes de referencia
print("Loading reference summaries...")
ref_cases = load_legal_data(train_ref_path)
ref_dict = {case.get("ID", ""): case.get("Summary", "") for case in ref_cases}

print(f"Testing with {len(test_cases)} cases")

# Crear el grafo
app = create_twenty_stage_pipeline_graph()
print("20-stage pipeline graph created successfully")

# Procesar casos de prueba
results = []
failed_cases = []
evaluation_results = []

for i, case in enumerate(test_cases):
    print(f"\nProcessing case {i+1}/{len(test_cases)}: {case.get('ID', 'unknown')}")
    
    result = process_single_case(case, app)
    results.append(result)
    
    if result["CurrentStage"] in ["error", "fallback_applied"]:
        failed_cases.append(result["ID"])
    
    print(f"   Status: {result['CurrentStage']}")
    if result["Error"]:
        print(f"   Error: {result['Error']}")
    
    # Evaluar si hay resumen de referencia disponible
    case_id = case.get("ID", "")
    if case_id in ref_dict and result["Summary"] != "Failed to generate summary":
        ref_summary = ref_dict[case_id]
        eval_result = evaluate_summary(result["Summary"], ref_summary)
        evaluation_results.append(eval_result)
        print(f"   ROUGE-2: {eval_result['rouge2_f']:.3f}, ROUGE-L: {eval_result['rougeL_f']:.3f}, BLEU: {eval_result['bleu']:.3f}")

# Calcular métricas promedio
if evaluation_results:
    avg_results, _ = evaluate_multiple_summaries(
        [r["Summary"] for r in results if r["Summary"] != "Failed to generate summary"],
        [ref_dict.get(case.get("ID", ""), "") for case in test_cases if case.get("ID", "") in ref_dict]
    )
    
    print(f"\nEVALUATION METRICS:")
    print(f"   • ROUGE-2: {avg_results['rouge2_f_mean']:.3f} ± {avg_results['rouge2_f_std']:.3f}")
    print(f"   • ROUGE-L: {avg_results['rougeL_f_mean']:.3f} ± {avg_results['rougeL_f_std']:.3f}")
    print(f"   • BLEU: {avg_results['bleu_mean']:.3f} ± {avg_results['bleu_std']:.3f}")
    
    print(f"\nAVERAGE SCORES (SUMMARY):")
    print(f"   • Average ROUGE-2: {avg_results['rouge2_f_mean']:.3f}")
    print(f"   • Average ROUGE-L: {avg_results['rougeL_f_mean']:.3f}")
    print(f"   • Average BLEU: {avg_results['bleu_mean']:.3f}")
    
    # Calcular promedio final
    final_average = (avg_results['rouge2_f_mean'] + avg_results['rougeL_f_mean'] + avg_results['bleu_mean']) / 3
    print(f"\nFINAL AVERAGE SCORE:")
    print(f"   • Final Average: {final_average:.3f}")

# Guardar resultados
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    for result in results:
        json.dump({"ID": result["ID"], "Summary": result["Summary"]}, f, ensure_ascii=False)
        f.write("\n")

print(f"\nResults saved to {output_path}")
print(f"Summary: {len(results)} total, {len(failed_cases)} failed")

print("\n20-Stage LangGraph Legal Pipeline (Training Evaluation) completed!")
print(f"   • Output: {output_path}")
print(f"   • Total cases: {len(results)}")
print(f"   • Failed cases: {len(failed_cases)}")
if failed_cases:
    print(f"   • Failed IDs: {failed_cases[:5]}{'...' if len(failed_cases) > 5 else ''}")

print(f"\nTRAINING EVALUATION RESULTS:")
print(f"   • Success Rate: {((len(results) - len(failed_cases)) / len(results) * 100):.1f}%")
print(f"   • Features: 20 Sequential Stages, Section-specific Extraction, Comprehensive Synthesis")

if evaluation_results:
    print(f"   • Average ROUGE-2: {avg_results['rouge2_f_mean']:.3f}")
    print(f"   • Average ROUGE-L: {avg_results['rougeL_f_mean']:.3f}")
    print(f"   • Average BLEU: {avg_results['bleu_mean']:.3f}")
    
    # Calcular y mostrar promedio final
    final_average = (avg_results['rouge2_f_mean'] + avg_results['rougeL_f_mean'] + avg_results['bleu_mean']) / 3
    print(f"   • Final Average Score: {final_average:.3f}")